# Practical 11 — Diabetes Prediction using KNN

Standalone solution notebook for **ML Practical 11**. Run cells from top to bottom and place the required dataset CSV in the working directory.https://www.kaggle.com/datasets/abdallamahgoub/diabetes

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, mean_squared_error, mean_absolute_error,
    r2_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

---
# Practical 11 — Diabetes Prediction using KNN
**Aim:** Use K-Nearest Neighbors on `diabetes.csv` and calculate confusion matrix, accuracy, error rate, precision, and recall.

Common dataset columns: `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`, and target `Outcome` (0 = no diabetes, 1 = diabetes). Some copies call the target `diabetes`.

In [ ]:
DIABETES_PATH = "diabetes.csv"
if not os.path.exists(DIABETES_PATH):
    alternatives = ["diabetes_data.csv", "diabetes_dataset.csv"]
    found = next((f for f in alternatives if os.path.exists(f)), None)
    if found:
        DIABETES_PATH = found
    else:
        raise FileNotFoundError(
            "Diabetes dataset not found. Put diabetes.csv beside this notebook and set DIABETES_PATH."
        )

diabetes = pd.read_csv(DIABETES_PATH)
diabetes.columns = [str(c).strip() for c in diabetes.columns]
print("Shape:", diabetes.shape)
display(diabetes.head())
print("Columns:", diabetes.columns.tolist())

In [ ]:
target_map = {str(c).strip().lower(): c for c in diabetes.columns}
DIABETES_TARGET = next((target_map[c] for c in ["outcome", "diabetes", "target", "class"]
                        if c in target_map), None)
if DIABETES_TARGET is None:
    raise ValueError("Could not identify diabetes target. Set DIABETES_TARGET manually.")

y = pd.to_numeric(diabetes[DIABETES_TARGET], errors="coerce")
X = diabetes.drop(columns=[DIABETES_TARGET]).apply(pd.to_numeric, errors="coerce")
data = pd.concat([X, y.rename("target")], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
X, y = data.drop(columns=["target"]), data["target"].astype(int)

# In the common Pima dataset, zero often encodes missing values for selected medical measurements.
for col in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]:
    if col in X.columns:
        X[col] = X[col].replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
diabetes_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train_scaled = diabetes_preprocessor.fit_transform(X_train)
X_test_scaled = diabetes_preprocessor.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

knn_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])
param_grid = {"knn__n_neighbors": list(range(3, 16, 2))}
grid = GridSearchCV(
    knn_pipeline, param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="accuracy", n_jobs=-1
)
grid.fit(X_train, y_train)
knn_best = grid.best_estimator_
y_pred = knn_best.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Best parameters:", grid.best_params_)
print(f"Accuracy: {accuracy:.4f}")
print(f"Error rate: {error_rate:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Diabetes KNN — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**Result:** KNN is trained using scaled features, the number of neighbors is tuned with 5-fold cross-validation, and the required classification metrics are displayed.